In [0]:
notes_test = spark.read.table("...")
notes_train = spark.read.table("...")

In [0]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"   
MAX_LENGTH = 512                                 
EPOCHS      = 4
BATCH_SIZE  = 4
GRAD_ACC    = 2         
FP16        = True      
LR          = 2e-5
WARMUP      = 0.06

import os, random, math, gc
import pandas as pd
from collections import defaultdict
from tabulate import tabulate

import torch
from torch.utils.data import DataLoader

from pyspark.sql import SparkSession
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    AdamW,
)
from sklearn.metrics import precision_recall_fscore_support

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

LABEL_COLUMNS = [
    "empathic_opportunity",
    "statement_of_emotion",
    "valence_negative",
    "valence_positive",
    "statement_of_progress",
    "statement_of_challenge",
]
NUM_LABELS = len(LABEL_COLUMNS)
TRAIN_SIZES = [10, 25, 50, 100, 200, 400, 500]
REPS = {s: (1 if s == 500 else 20) for s in TRAIN_SIZES}


spark = SparkSession.builder.getOrCreate()

try:                                
    notes_train_df = notes_train.toPandas()
    notes_test_df  = notes_test.toPandas()
except NameError:
    notes_train_df = spark.table("notes_train").toPandas()
    notes_test_df  = spark.table("notes_test").toPandas()


# Build  Datasets 

def make_hf_dataset(pdf: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(
        pdf[["message_text"] + LABEL_COLUMNS],
        preserve_index=False,
    )
    def stack(example):
        example["labels"] = [float(example[c]) for c in LABEL_COLUMNS]
        return example
    ds = ds.map(stack, remove_columns=LABEL_COLUMNS)
    return ds

hf_train_full = make_hf_dataset(notes_train_df)
hf_test_full  = make_hf_dataset(notes_test_df)


# Tokeniser

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(batch):
    return tokenizer(batch["message_text"], truncation=True, max_length=MAX_LENGTH)

hf_test = hf_test_full.map(tok, batched=True)
hf_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)


# Metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred        
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = labels.astype(int)

    micro = precision_recall_fscore_support(labels, preds, average="micro", zero_division=0)
    macro = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    per   = precision_recall_fscore_support(labels, preds, average=None , zero_division=0)

    metrics = {
        "micro_precision": micro[0], "micro_recall": micro[1], "micro_f1": micro[2],
        "macro_precision": macro[0], "macro_recall": macro[1], "macro_f1": macro[2],
    }
    for i, lbl in enumerate(LABEL_COLUMNS):
        metrics[f"{lbl}_precision"] = per[0][i]
        metrics[f"{lbl}_recall"]    = per[1][i]
        metrics[f"{lbl}_f1"]        = per[2][i]
    return metrics


# Training loop

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results_store = defaultdict(list)

for size in TRAIN_SIZES:
    for rep in range(REPS[size]):
        seed = 2025 + rep
        torch.manual_seed(seed); random.seed(seed)

        if size == 500:
            hf_train = hf_train_full
        else:
            idx = random.sample(range(len(hf_train_full)), size)
            hf_train = hf_train_full.select(idx)

        hf_train = hf_train.map(tok, batched=True)
        hf_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=NUM_LABELS,
            problem_type="multi_label_classification",
        ).to(DEVICE)

        training_args = TrainingArguments(
            output_dir=f"./runs/{MODEL_NAME.replace('/','_')}_{size}_{rep}",
            overwrite_output_dir=True,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACC,
            num_train_epochs=EPOCHS,
            learning_rate=LR,
            warmup_ratio=WARMUP,
            weight_decay=0.01,
            fp16=FP16,
            optim="adamw_torch_fused",
            logging_steps=10,
            evaluation_strategy="no",
            save_strategy="no",
            report_to=[],
            seed=seed,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=hf_train,
            eval_dataset=hf_test,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        metrics = trainer.evaluate()

        for k, v in metrics.items():
            results_store[(size, k)].append(v)

        # cleanup
        del model, trainer
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()



def mean_ci(vals):
    n = len(vals); m = sum(vals)/n
    if n == 1: return m, 0.0
    sd = math.sqrt(sum((x-m)**2 for x in vals) / (n-1))
    ci = 1.96*sd/math.sqrt(n)
    return m, ci

rows = []
all_metrics = sorted({m for (_, m) in results_store.keys()})
for size in TRAIN_SIZES:
    row = {"train_size": size}
    for m in all_metrics:
        mean, ci = mean_ci(results_store[(size,m)])
        row[m] = f"{mean:.4f} ± {ci:.4f}"
    rows.append(row)

pdf = pd.DataFrame(rows).set_index("train_size")

results_spark_df = spark.createDataFrame(pdf.reset_index())
results_spark_df.show(truncate=False)        

In [0]:
display(results_spark_df)